- 首先我们会从 huggingface上面去加载数据集 -> 要么翻墙，要么镜像
- 其次, autodl扩容的是数据盘， 我们存下来的要保存到数据盘


In [3]:
%env HF_ENDPOINT=https://hf-mirror.com
%env HF_HOME=/root/autodl-tmp/hf
!pip install trl datasets

env: HF_ENDPOINT=https://hf-mirror.com
env: HF_HOME=/root/autodl-tmp/hf
Looking in indexes: http://mirrors.aliyun.com/pypi/simple


- 选一个小参数模型

In [4]:
# 加载model和Tokenizer
from transformers import AutoModelForCausalLM,AutoTokenizer

model_name = 'Qwen/Qwen3-0.6B'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

model.safetensors:  79%|#######8  | 1.18G/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

- 我们选用的是Instruct model, 那么数据集格式就应该参照open-ai 那种一问一答的形式

In [5]:
# 数据集加载以及处理
from datasets import load_dataset
# 传入训练集、测试集
dataset_dict = load_dataset('json',data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})

# 将数据集格式转成trl的格式
def map_func(example):
    conversation = example['conversation']
    messages = []
    for item in conversation:
        messages.append({'role':'user','content':item['human']})
        messages.append({'role':'assistant','content':item['assistant']})
    return {'messages':messages}

dataset_dict = dataset_dict.map(map_func,batched=False,remove_columns=['dataset','conversation','category','conversation_id'])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/49500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [6]:
# 检查一下
dataset_dict["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

### 训练参数修改
- 训练结果路径
- 日志目录 (用autodl 默认的 tensor board目录 - /root/tf-logs)
- save_total_limit=2  <-保存最近两次, 方便断点续传


In [10]:
import os

from trl import SFTConfig, SFTTrainer

# TensorBoard：Transformers 5.x 用 TENSORBOARD_LOGGING_DIR；logging_dir 参数已废弃且不会写入该路径
# 必须在实例化 SFTTrainer 之前设置，以便 TensorBoardCallback 读到
os.environ["TENSORBOARD_LOGGING_DIR"] = "/root/tf-logs"

# Configure trainer 训练器设置
# 改一下存的数据路径
training_args = SFTConfig(
    output_dir="/root/autodl-tmp/.autodl/sft/Qwen3-0.6B/sft-full",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    report_to="tensorboard",
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50
)

# Initialize trainer 初始化训练器
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer
)

In [11]:
# 察看数据集处理结果
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))
print(tokenizer.decode(batch['input_ids'][0]))

<|im_start|>user
关键词抽取：
以手动换挡机构疲劳寿命试验为目的,构建了一种模拟驾驶员进行选档、换挡操作的试验平台,该平台集成二自由度运动滑台与气动加载装置为一体形成换挡运动加载机构.以该机构为研究对象,通过建立换挡与选挡的运动轨迹模型,分析换挡运动加载机构位移输出与运动轨迹之间的关系,通过分析换挡机构操纵杆受力情况,分别对换挡动作和选档动作进行力学分析,并得出加载力的计算方法.最后结合电气控制技术与气动控制技术,对系统进行了试验,结果表明,系统具有可行性与正确性.<|im_end|>
<|im_start|>assistant
<think>

</think>

换挡机构;试验平台;疲劳性能<|im_end|>
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [12]:
trainer.train()

Step,Training Loss,Validation Loss
100,1.932202,2.608608
200,1.858552,2.608761
300,2.200657,2.578224
400,2.147539,2.556095
500,2.280823,2.523661
600,2.225193,2.498091
700,2.510765,2.477617
800,2.426313,2.467745
900,2.487849,2.463192
1000,2.497931,2.462255


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=2.244687114715576, metrics={'train_runtime': 269.677, 'train_samples_per_second': 14.833, 'train_steps_per_second': 3.708, 'total_flos': 3786943561728000.0, 'train_loss': 2.244687114715576})

- 保存一下模型

In [15]:
trainer.save_model('/root/autodl-tmp/.autodl/sft/Qwen3-0.6B/sft-full/best')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
next(model.parameters()).dtype

torch.bfloat16